Copyright © 2026 [Giovanni Squillero](https://squillero.github.io/) / [Politecnico di Torino](https://www.polito.it)  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE) for details.  

![](i/susi2.webp)

| Original Italian                                                                                                        | English Translation                                                                                                                |
| ----------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| Susi, in questo palazzo di tre piani hanno il loro studio tre professionisti, che si chiamano Renato, Sandro e Tommaso. | Susi, in this three-story building, three professionals have their offices. Their names are Renato, Sandro, and Tommaso.           |
| Uno è architetto, uno è medico e uno è ingegnere.                                                                       | One is an architect, one is a doctor, and one is an engineer.                                                                      |
| Ognuno ha una segretaria personale. Si chiamano Anna, Bianca e Carola.                                                  | Each has a personal secretary. Their names are Anna, Bianca, and Carola.                                                           |
| All'ora di pranzo di solito Anna sale dalla segretaria di Sandro per mangiare con lei, mentre Bianca pranza al bar.     | At lunchtime, Anna usually goes up to Sandro's secretary's office to eat with her, while Bianca has lunch at the café.             |
| Hanno tutti lo studio a un piano diverso.                                                                               | They all have their offices on different floors.                                                                                   |
| L'ingegnere è al primo.                                                                                                 | The engineer is on the first floor.                                                                                                |
| Renato, quando a fine giornata scende per andare a casa, entra sempre dall'architetto per un rapido saluto.             | At the end of the day, when Renato goes downstairs to go home, Renato always stops by the architect's office for a quick greeting. |
| Susi, a questo punto puoi dirmi a che piano lavora Carola?                                                              | Susi, can you now tell me on which floor Carola works?                                                                             |

In [1]:
import logging
from itertools import permutations
from icecream import ic

In [2]:
# Both the RiSEs' z3 binary and the z3-solver Python package are needed.
# see: https://en.wikipedia.org/wiki/Z3_Theorem_Prover
import z3

In [3]:
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("Susi")

In [4]:
profession, (architect, doctor, engineer) = z3.EnumSort(
    "professions", ["architect", "doctor", "engineer"]
)
all_professions = {architect, doctor, engineer}
professional, (renato, sandro, tommaso) = z3.EnumSort(
    "professionals", ["Renato", "Sandro", "Tommaso"]
)
all_professionals = {renato, sandro, tommaso}
secretary, (anna, bianca, carola) = z3.EnumSort("secretaries", ["Anna", "Bianca", "Carola"])
all_secretaries = {anna, bianca, carola}

In [5]:
solver = z3.Solver()

# Map professional <-> profession
job = z3.Function("job", professional, profession)
solver.add(z3.Distinct(job(renato), job(sandro), job(tommaso)))

# A Pseudo-Boolean (PbEq) equality constraint expresses that a weighted sum of Boolean
# variables must equal a specific value.
# Here, all weights == 1, and the sum == 1. Ie. each professional hires excatly 1 secretary.
hires = z3.Function("hires", professional, secretary, z3.BoolSort())
for p in all_professionals:
    solver.add(z3.PbEq([(hires(p, s), 1) for s in all_secretaries], 1))
for s in all_secretaries:
    solver.add(z3.PbEq([(hires(p, s), 1) for p in all_professionals], 1))

# Each professional works in a office on a distinct floor (1, 2, or 3)
floor_p = z3.Function("floor_p", professional, z3.IntSort())
for p in all_professionals:
    solver.add(z3.And(floor_p(p) >= 1, floor_p(p) <= 3))
# All on different floors
solver.add(z3.Distinct(*[floor_p(p) for p in all_professionals]))

# Each secreatry works in a office on a distinct floor (1, 2, or 3)
floor_s = z3.Function("floor_s", secretary, z3.IntSort())
for s in all_secretaries:
    solver.add(z3.And(floor_s(s) >= 1, floor_s(s) <= 3))
# All on different floors
solver.add(z3.Distinct(*[floor_s(s) for s in all_secretaries]))

# Professional & secretary are in the same offic (ie. same floor)
for p in all_professionals:
    for s in all_secretaries:
        solver.add(z3.Implies(hires(p, s), floor_p(p) == floor_s(s)))

In [6]:
# The engineer is on floor 1
for p in all_professionals:
    solver.add(z3.Implies(job(p) == engineer, floor_p(p) == 1))

# The professional who hires Anna stays on a lower floor than Sandro
solver.add(floor_s(anna) < floor_p(sandro))

# Sandro hires Carola (not Anna, nor Bianca)
solver.add(hires(sandro, carola))

# Renato's floor > architect floor
for p in all_professionals:
    solver.add(z3.Implies(job(p) == architect, floor_p(renato) > floor_p(p)))

In [7]:
solver.assertions()

[Distinct(job(Renato), job(Sandro), job(Tommaso)),
 PbEq(((hires(Sandro, Carola), 1),
       (hires(Sandro, Anna), 1),
       (hires(Sandro, Bianca), 1)),
      1),
 PbEq(((hires(Tommaso, Carola), 1),
       (hires(Tommaso, Anna), 1),
       (hires(Tommaso, Bianca), 1)),
      1),
 PbEq(((hires(Renato, Carola), 1),
       (hires(Renato, Anna), 1),
       (hires(Renato, Bianca), 1)),
      1),
 PbEq(((hires(Sandro, Carola), 1),
       (hires(Tommaso, Carola), 1),
       (hires(Renato, Carola), 1)),
      1),
 PbEq(((hires(Sandro, Anna), 1),
       (hires(Tommaso, Anna), 1),
       (hires(Renato, Anna), 1)),
      1),
 PbEq(((hires(Sandro, Bianca), 1),
       (hires(Tommaso, Bianca), 1),
       (hires(Renato, Bianca), 1)),
      1),
 And(floor_p(Sandro) >= 1, floor_p(Sandro) <= 3),
 And(floor_p(Tommaso) >= 1, floor_p(Tommaso) <= 3),
 And(floor_p(Renato) >= 1, floor_p(Renato) <= 3),
 Distinct(floor_p(Sandro),
          floor_p(Tommaso),
          floor_p(Renato)),
 And(floor_s(Carola) >= 1, floor_s(Carola) <= 3),
 And(floor_s(Anna) >= 1, floor_s(Anna) <= 3),
 And(floor_s(Bianca) >= 1, floor_s(Bianca) <= 3),
 Distinct(floor_s(Carola), floor_s(Anna), floor_s(Bianca)),
 Implies(hires(Sandro, Carola),
         floor_p(Sandro) == floor_s(Carola)),
 Implies(hires(Sandro, Anna),
         floor_p(Sandro) == floor_s(Anna)),
 Implies(hires(Sandro, Bianca),
         floor_p(Sandro) == floor_s(Bianca)),
 Implies(hires(Tommaso, Carola),
         floor_p(Tommaso) == floor_s(Carola)),
 Implies(hires(Tommaso, Anna),
         floor_p(Tommaso) == floor_s(Anna)),
 Implies(hires(Tommaso, Bianca),
         floor_p(Tommaso) == floor_s(Bianca)),
 Implies(hires(Renato, Carola),
         floor_p(Renato) == floor_s(Carola)),
 Implies(hires(Renato, Anna),
         floor_p(Renato) == floor_s(Anna)),
 Implies(hires(Renato, Bianca),
         floor_p(Renato) == floor_s(Bianca)),
 Implies(job(Sandro) == engineer, floor_p(Sandro) == 1),
 Implies(job(Tommaso) == engineer, floor_p(Tommaso) == 1),
 Implies(job(Renato) == engineer, floor_p(Renato) == 1),
 floor_s(Anna) < floor_p(Sandro),
 hires(Sandro, Carola),
 Implies(job(Sandro) == architect,
         floor_p(Renato) > floor_p(Sandro)),
 Implies(job(Tommaso) == architect,
         floor_p(Renato) > floor_p(Tommaso)),
 Implies(job(Renato) == architect,
         floor_p(Renato) > floor_p(Renato))]

In [8]:
if solver.check() == z3.sat:
    model = solver.model()
    ic(model)
    ic(model.evaluate(floor_s(carola)))

ic| model: [hires = [(Tommaso, Anna) -> True,
                     (Renato, Bianca) -> True,
                     (Sandro, Carola) -> True,
                     else -> False],
            job = [Sandro -> architect,
                   Tommaso -> engineer,
                   else -> doctor],
            floor_p = [Renato -> 3, Sandro -> 2, else -> 1],
            floor_s = [Bianca -> 3, Carola -> 2, else -> 1]]
ic| model.evaluate(floor_s(carola)): 2


In [9]:
solver.add(floor_s(carola) != model.evaluate(floor_s(carola)))
z3.set_param("verbose", 20)
solver.check()

unsat